In [ ]:
import time
from datetime import datetime

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

### redirect testing

In [52]:
from selenium import webdriver
from selenium.common.exceptions import WebDriverException


def get_redirect_link(url):
    """
    Resolves a redirect URL to its final destination URL using Selenium.

    Args:
      url: The initial URL that may be a redirect.

    Returns:
      The final destination URL after following redirects, or the original URL
      if an error occurred.
    """
    # Set up the WebDriver (e.g., ChromeDriver). Make sure the driver executable
    # is in your system's PATH or provide the path to the executable.
    # You might need to change this based on the browser you have installed (e.g., Firefox, Edge).
    driver = None  # Initialize driver to None
    try:
        # Using Chrome as an example. You might need options like --headless
        options = webdriver.ChromeOptions()
        # if you don't want a browser window to open.
        # Uncomment the line below to run in headless mode (no browser window)
        options.add_argument("--headless")
        # options.add_argument('--no-sandbox') # Recommended for some environments
        # options.add_argument('--disable-dev-shm-usage') # Recommended for some environments

        driver = webdriver.Chrome(options=options)
        # print(f"Attempting to resolve URL: {url}")
        driver.get(url)

        # Wait for the body element to be present. This is a more reliable indicator
        # that the page has loaded after redirects compared to checking the title.
        # print("sleeping for 1 seconds to allow for any additional redirects...")
        time.sleep(1)
        # Stop the driver from loading further by executing a script to stop network activity
        # print("Stopping the driver from loading further...")
        driver.execute_script("window.stop();")
        # Get the current URL after all redirects have occurred
        final_url = driver.current_url
        # print(f"Final URL resolved by Selenium: {final_url}")

        return final_url

    except WebDriverException:
        # print(f"An error occurred with Selenium: {e}")
        return url  # Return original URL if an error occurs
    finally:
        # Always close the browser session if the driver was successfully initialized
        if driver:
            driver.quit()
            # print("Browser session closed.")

In [53]:
url = "https://news.google.com/read/CBMidEFVX3lxTE5naFB0MVFURFlfUE00aFZTN0FkeThUT0pqbkpZREdjeXBGck1MVE9SWWlmcHhiSUtmc3VEekIxQVJpS21HUlBNT0s5R1pESDNCSmJsQmFCVmRjNVhTVjBzOFBQcTRyUi1tc1dfdTltV0pvUXhL?hl=en-US&gl=US&ceid=US%3Aen"

In [ ]:
# Example usage:
redirect_url_selenium = url  # Replace with your redirect URL
final_url_selenium = get_redirect_link(redirect_url_selenium)

if final_url_selenium != redirect_url_selenium:
    print(f"The original redirect URL was: {redirect_url_selenium}")
    print(f"The final destination URL is: {final_url_selenium}")
else:
    print(f"\nNo significant redirect occurred or an error happened for URL: {redirect_url_selenium}")


The original redirect URL was: https://news.google.com/read/CBMidEFVX3lxTE5naFB0MVFURFlfUE00aFZTN0FkeThUT0pqbkpZREdjeXBGck1MVE9SWWlmcHhiSUtmc3VEekIxQVJpS21HUlBNT0s5R1pESDNCSmJsQmFCVmRjNVhTVjBzOFBQcTRyUi1tc1dfdTltV0pvUXhL?hl=en-US&gl=US&ceid=US%3Aen
The final destination URL is: https://www.gbtribune.com/news/business/kansas-canola-fields-gold/


### Single iteration

In [30]:
search = "canola"
start_date = "2024-04-04"
end_date = "2024-04-05"


url = f"https://news.google.com/search?q={search}%20after%3A{start_date}%20before%3A{end_date}&hl=en-US&gl=US&ceid=US%3Aen"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html")

In [50]:
temp = soup.find_all("a", {"class": "JtKRv"})
times = soup.find_all("time", {"class": "hvbAAd"})
data = []
for i in range(len(temp)):
    link = "https://news.google.com/" + temp[i]["href"][2:]
    title = temp[i].text
    date = times[i]["datetime"]
    data.append((title, date, link))

df = pd.DataFrame(data, columns=["title", "date", "link"])
df

,title,date,link
0,Kansas Canola: Fields of gold,2024-04-05T07:00:00Z,https://news.google.com/read/CBMidEFVX3lxTE5na...
1,Pristine landscapes at the core: Pursuing a ne...,2024-04-05T07:00:00Z,https://news.google.com/read/CBMiWkFVX3lxTE8zR...
2,Concerns raised that U.S. soy crush capacity i...,2024-04-04T07:00:00Z,https://news.google.com/read/CBMilAFBVV95cUxNL...
3,Tolvera Herbicide Offers New Active Ingredient...,2024-04-05T07:00:00Z,https://news.google.com/read/CBMimgFBVV95cUxOU...
4,Do You Know How Flavor Works? (Published 2024),2024-04-04T07:00:00Z,https://news.google.com/read/CBMiekFVX3lxTE1jR...
5,Feedgrain Focus: Prices jump after wet week fo...,2024-04-04T07:00:00Z,https://news.google.com/read/CBMilwFBVV95cUxNa...
6,Workplace Safety,2024-04-04T07:00:00Z,https://news.google.com/read/CBMiXEFVX3lxTE40U...
7,Ministry of Labour focuses on summer students ...,2024-04-04T07:00:00Z,https://news.google.com/read/CBMipgFBVV95cUxNd...
8,Morris 9365 does a great job for this Bolgart ...,2024-04-04T07:00:00Z,https://news.google.com/read/CBMi9AFBVV95cUxQd...
9,8 Non-Dairy Milk Brands You Should Think Twice...,2024-04-04T07:00:00Z,https://news.google.com/read/CBMic0FVX3lxTE9pd...


### Multiple Iteration

In [ ]:
search = "canola"
start_date = "2024-04-04"
end_date = "2024-04-30"

start = datetime.strptime(start_date, "%Y-%m-%d")
end = datetime.strptime(end_date, "%Y-%m-%d")
current = start
runs = (end - start).days
pbar = tqdm(total=runs + 1)

  0%|          | 0/2 [00:07<?, ?it/s]


In [ ]:
from data_collection import fetch_news


search = "canola"
start_date = "2024-04-10"
end_date = "2024-04-15"

df = fetch_news(search, start_date, end_date)

  0%|          | 0/6 [00:00<?, ?it/s]

Fetching news articles for canola from 2024-04-10 to 2024-04-15...


100%|██████████| 6/6 [00:04<00:00,  1.32it/s]


In [64]:
temp = pd.read_csv("canola_news_2024-01-01_to_2025-04-24.csv")
temp.head()

,Date,Timestamp,Title,Description,Source Name,Source URL,Article URL,Query
0,2025-04-23,2025-04-23T16:15:50,Ag markets find new trade levels following tar...,Ag markets find new trade levels following tar...,The Western Producer,https://www.producer.com,https://news.google.com/rss/articles/CBMilAFBV...,"""canola"""
1,2025-04-23,2025-04-23T15:30:35,Quick crop establishment lowers risk of flea b...,Quick crop establishment lowers risk of flea b...,Farms.com,https://m.farms.com,https://news.google.com/rss/articles/CBMikwFBV...,"""canola"""
2,2025-04-23,2025-04-23T05:00:12,From olive to canola: Here’s how many calories...,From olive to canola: Here’s how many calories...,The Indian Express,https://indianexpress.com,https://news.google.com/rss/articles/CBMirwFBV...,"""canola"""
3,2025-04-23,2025-04-23T19:47:14,ICE canola futures continue rise on low stocks...,ICE canola futures continue rise on low stocks...,TradingView,https://www.tradingview.com,https://news.google.com/rss/articles/CBMiwwFBV...,"""canola"""
4,2025-04-23,2025-04-23T20:32:27,ICE Canada Weekly: Canola in a good position -...,ICE Canada Weekly: Canola in a good position ...,Alberta Farmer Express,https://www.albertafarmexpress.ca,https://news.google.com/rss/articles/CBMijAFBV...,"""canola"""
